# Epochs — Audio Feature Extraction

Run a small pilot first, then full extraction. Do not start the full 8,000-track run until the pilot is verified.

In [1]:
from pathlib import Path
import sys, time
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "processed"
AUDIO_DIR = PROJECT_ROOT / "data" / "extracted" / "fma_small"

sys.path.insert(0, str(PROJECT_ROOT))
from src.feature_extraction import extract_features

metadata = pd.read_csv(DATA_DIR / "fma_track_metadata.csv")

def fma_audio_path(track_id):
    track_id = int(track_id)
    return AUDIO_DIR / f"{track_id // 1000:03d}" / f"{track_id:06d}.mp3"

metadata["audio_path"] = metadata["track_id"].map(fma_audio_path)
metadata["audio_exists"] = metadata["audio_path"].map(Path.exists)

print("Audio found:", int(metadata["audio_exists"].sum()))
print("Audio missing:", int((~metadata["audio_exists"]).sum()))


Audio found: 8000
Audio missing: 0


In [2]:
PILOT_PER_GENRE = 5
pilot = (
    metadata[metadata["audio_exists"]]
    .dropna(subset=["genre"])
    .groupby("genre", group_keys=False)
    .head(PILOT_PER_GENRE)
    .copy()
)
print("Pilot tracks:", len(pilot))
print(pilot["genre"].value_counts())


Pilot tracks: 40
genre
Hip-Hop          5
Pop              5
Folk             5
Experimental     5
Rock             5
International    5
Electronic       5
Instrumental     5
Name: count, dtype: int64


In [3]:
rows, failures = [], []

for _, row in pilot.iterrows():
    try:
        features = extract_features(row["audio_path"])
        features.update({
            "track_id": int(row["track_id"]),
            "artist_id": int(row["artist_id"]),
            "genre": row["genre"],
        })
        rows.append(features)
    except Exception as exc:
        failures.append({"track_id": int(row["track_id"]), "error": repr(exc)})

pilot_features = pd.DataFrame(rows)
print("Successful:", len(pilot_features))
print("Failures:", len(failures))
print("Shape:", pilot_features.shape)
print("Columns:", pilot_features.columns.tolist())


Successful: 40
Failures: 0
Shape: (40, 64)
Columns: ['mfcc_1_mean', 'mfcc_1_std', 'mfcc_2_mean', 'mfcc_2_std', 'mfcc_3_mean', 'mfcc_3_std', 'mfcc_4_mean', 'mfcc_4_std', 'mfcc_5_mean', 'mfcc_5_std', 'mfcc_6_mean', 'mfcc_6_std', 'mfcc_7_mean', 'mfcc_7_std', 'mfcc_8_mean', 'mfcc_8_std', 'mfcc_9_mean', 'mfcc_9_std', 'mfcc_10_mean', 'mfcc_10_std', 'mfcc_11_mean', 'mfcc_11_std', 'mfcc_12_mean', 'mfcc_12_std', 'mfcc_13_mean', 'mfcc_13_std', 'chroma_1_mean', 'chroma_1_std', 'chroma_2_mean', 'chroma_2_std', 'chroma_3_mean', 'chroma_3_std', 'chroma_4_mean', 'chroma_4_std', 'chroma_5_mean', 'chroma_5_std', 'chroma_6_mean', 'chroma_6_std', 'chroma_7_mean', 'chroma_7_std', 'chroma_8_mean', 'chroma_8_std', 'chroma_9_mean', 'chroma_9_std', 'chroma_10_mean', 'chroma_10_std', 'chroma_11_mean', 'chroma_11_std', 'chroma_12_mean', 'chroma_12_std', 'spectral_centroid_mean', 'spectral_centroid_std', 'spectral_bandwidth_mean', 'spectral_bandwidth_std', 'spectral_rolloff_mean', 'spectral_rolloff_std', 'zcr_me

## Full extraction — run only after the pilot is correct.

In [5]:
FULL_OUTPUT = DATA_DIR / "fma_features.csv"

existing = set()
if FULL_OUTPUT.exists():
    existing_df = pd.read_csv(FULL_OUTPUT, usecols=["track_id"])
    existing = set(existing_df["track_id"].astype(int))

rows, failures = [], []
start = time.time()

for i, row in enumerate(metadata[metadata["audio_exists"]].itertuples(index=False), start=1):
    track_id = int(row.track_id)
    if track_id in existing:
        continue
    try:
        features = extract_features(row.audio_path)
        features.update({
            "track_id": track_id,
            "artist_id": int(row.artist_id),
            "genre": row.genre,
        })
        rows.append(features)
    except Exception as exc:
        failures.append({"track_id": track_id, "error": repr(exc)})
    if i % 100 == 0:
        print(f"Processed metadata rows: {i}")

new_df = pd.DataFrame(rows)
if FULL_OUTPUT.exists() and not new_df.empty:
    combined = pd.concat([pd.read_csv(FULL_OUTPUT), new_df], ignore_index=True)
else:
    combined = new_df

combined.to_csv(FULL_OUTPUT, index=False)

print("Saved:", FULL_OUTPUT)
print("Rows:", len(combined))
print("Failures:", len(failures))
print(f"Elapsed: {(time.time()-start)/60:.1f} minutes")


Processed metadata rows: 100
Processed metadata rows: 200
Processed metadata rows: 300
Processed metadata rows: 400
Processed metadata rows: 500
Processed metadata rows: 600
Processed metadata rows: 700
Processed metadata rows: 800
Processed metadata rows: 900
Processed metadata rows: 1000
Processed metadata rows: 1100
Processed metadata rows: 1200
Processed metadata rows: 1300
Processed metadata rows: 1400
Processed metadata rows: 1500
Processed metadata rows: 1600


d:\ML-clgProject\Epochs-Music-Genre-Classifier\mvenv\lib\site-packages\librosa\core\pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(


Processed metadata rows: 1700
Processed metadata rows: 1800
Processed metadata rows: 1900
Processed metadata rows: 2000
Processed metadata rows: 2100
Processed metadata rows: 2200
Processed metadata rows: 2300
Processed metadata rows: 2400
Processed metadata rows: 2500
Processed metadata rows: 2600
Processed metadata rows: 2700
Processed metadata rows: 2800
Processed metadata rows: 2900
Processed metadata rows: 3000
Processed metadata rows: 3100
Processed metadata rows: 3200
Processed metadata rows: 3300
Processed metadata rows: 3400
Processed metadata rows: 3500
Processed metadata rows: 3600
Processed metadata rows: 3700
Processed metadata rows: 3800
Processed metadata rows: 3900
Processed metadata rows: 4000
Processed metadata rows: 4100
Processed metadata rows: 4200
Processed metadata rows: 4300
Processed metadata rows: 4400


d:\ML-clgProject\Epochs-Music-Genre-Classifier\src\audio_processing.py:9: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(path, sr=sr, mono=True, duration=duration)
d:\ML-clgProject\Epochs-Music-Genre-Classifier\mvenv\lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Processed metadata rows: 4500
Processed metadata rows: 4600
Processed metadata rows: 4700
Processed metadata rows: 4800
Processed metadata rows: 4900
Processed metadata rows: 5000
Processed metadata rows: 5100
Processed metadata rows: 5200
Processed metadata rows: 5300
Processed metadata rows: 5400
Processed metadata rows: 5500
Processed metadata rows: 5600
Processed metadata rows: 5700
Processed metadata rows: 5800
Processed metadata rows: 5900
Processed metadata rows: 6000
Processed metadata rows: 6100
Processed metadata rows: 6200
Processed metadata rows: 6300
Processed metadata rows: 6400
Processed metadata rows: 6500
Processed metadata rows: 6600
Processed metadata rows: 6700
Processed metadata rows: 6800
Processed metadata rows: 6900
Processed metadata rows: 7000
Processed metadata rows: 7100
Processed metadata rows: 7200
Processed metadata rows: 7300
Processed metadata rows: 7400
Processed metadata rows: 7500
Processed metadata rows: 7600
Processed metadata rows: 7700
Processed 

In [6]:
features = pd.read_csv(FULL_OUTPUT)
print("Shape:", features.shape)
print("Duplicate track IDs:", features["track_id"].duplicated().sum())
print("Missing values:", int(features.isna().sum().sum()))
print(features["genre"].value_counts())


Shape: (7997, 64)
Duplicate track IDs: 0
Missing values: 0
genre
Hip-Hop          1000
Pop              1000
Folk             1000
International    1000
Instrumental     1000
Experimental      999
Rock              999
Electronic        999
Name: count, dtype: int64


In [7]:
print("Failure count:", len(failures))

for failure in failures:
    print(failure)

Failure count: 3
{'track_id': 99134, 'error': 'NoBackendError()'}
{'track_id': 108925, 'error': 'NoBackendError()'}
{'track_id': 133297, 'error': 'NoBackendError()'}


In [10]:
from pathlib import Path

AUDIO_DIR = Path("../data/extracted/fma_small")

for track_id in [99134, 108925, 133297]:
    path = AUDIO_DIR / f"{track_id // 1000:03d}" / f"{track_id:06d}.mp3"
    print(track_id, path, path.exists(), path.stat().st_size if path.exists() else None)

99134 ..\data\extracted\fma_small\099\099134.mp3 True 1298
108925 ..\data\extracted\fma_small\108\108925.mp3 True 1371
133297 ..\data\extracted\fma_small\133\133297.mp3 True 1752


In [11]:
for track_id in [99134, 108925, 133297]:
    path = AUDIO_DIR / f"{track_id // 1000:03d}" / f"{track_id:06d}.mp3"
    print(track_id, path, path.exists(), path.stat().st_size if path.exists() else None)

99134 ..\data\extracted\fma_small\099\099134.mp3 True 1298
108925 ..\data\extracted\fma_small\108\108925.mp3 True 1371
133297 ..\data\extracted\fma_small\133\133297.mp3 True 1752


In [12]:
import soundfile as sf

for track_id in [99134, 108925, 133297]:
    path = AUDIO_DIR / f"{track_id // 1000:03d}" / f"{track_id:06d}.mp3"
    try:
        info = sf.info(path)
        print(track_id, "OK:", info)
    except Exception as e:
        print(track_id, "FAILED:", type(e).__name__, str(e))

99134 FAILED: LibsndfileError Error opening '..\\data\\extracted\\fma_small\\099\\099134.mp3': File does not exist or is not a regular file (possibly a pipe?).
108925 FAILED: LibsndfileError Error opening '..\\data\\extracted\\fma_small\\108\\108925.mp3': File does not exist or is not a regular file (possibly a pipe?).
133297 FAILED: LibsndfileError Error opening '..\\data\\extracted\\fma_small\\133\\133297.mp3': File does not exist or is not a regular file (possibly a pipe?).
